## **MSE Error for properties for test samples and generated samples**

In [1]:
import sys, os

sys.path.append(os.path.abspath('..'))

### **Tokenizated data loading**

In [2]:
import json

with open('../Transformer/token_to_id.json', 'r') as f:
    token_to_id = json.load(f)
with open('../Transformer/id_to_token.json', 'r') as f:
    id_to_token = json.load(f)

### **Hyperparameters and model creation**

In [3]:
import tensorflow as tf
from Transformer.transformer import TransformerNoEnc
from Gumbel.gumbel import GumbelSoftmax
import numpy as np

num_layers = 2
d_model = 64
dff = 128
num_heads = 4
dropout = 0.1
SEQ_LEN = 25
vocab_size = len(token_to_id)

transformer_layer = TransformerNoEnc(
	num_layers=num_layers,
	d_model=d_model,
	num_heads=num_heads,
	dff=dff,
	seq_len=SEQ_LEN,
    vocab_size=vocab_size,
	dropout=dropout
)

transformer_corr_trained = TransformerNoEnc(
	num_layers=num_layers,
	d_model=d_model,
	num_heads=num_heads,
	dff=dff,
	seq_len=SEQ_LEN,
    vocab_size=vocab_size,
	dropout=dropout
)

props_dim =  23
tmp1, tmp2 = np.random.randn(1, props_dim), np.random.randn(1, 1)
tmp = transformer_layer((tmp1, tmp2))
tmp = transformer_corr_trained((tmp1, tmp2))

gumbel_softmax_layer = GumbelSoftmax(transformer_layer)
tmp = gumbel_softmax_layer((tmp1, tmp2), beta=1)

gen_input_context = tf.keras.layers.Input((props_dim, ))
gen_input_dec = tf.keras.layers.Input((None, ))
gumbel_softmax_layer_out = gumbel_softmax_layer((gen_input_context, gen_input_dec), beta=1)
generator = tf.keras.Model(inputs=(gen_input_context, gen_input_dec), outputs=gumbel_softmax_layer_out)

transformer_corr_trained.load_weights('../SavedWeights/corr_transformer_weights.weights.h5')
generator.load_weights('../SavedWeights/generator.weights.h5')


2026-05-08 00:05:55.964572: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778191563.569177    4169 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13584 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9
/home/ubuntu/miniconda3/envs/new_tensorflow_2/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'positional_encoding' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/ubuntu/miniconda3/envs/new_tensorflow_2/lib/python3.12/site-p

### **Data loading**

In [4]:
import tensorflow_datasets as tfds

ds = tfds.load('qm9', split='train')
ds = tfds.as_dataframe(ds)

features_to_drop = ['InChI', 'InChI_relaxed', 'SMILES_relaxed', 'tag', 'frequencies', 'charges', 'positions', 'Mulliken_charges', 'num_atoms', 'index']
ds = ds.drop(features_to_drop, axis=1)

ds_smiles = ds['SMILES'].astype('string')
ds_props = ds.drop(['SMILES'], axis=1)

2026-05-08 00:06:11.190085: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-05-08 00:07:00.502480: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### **Tokenizated data loading, train/test split and scaling**

In [5]:
from sklearn.model_selection import train_test_split
from utils_functions import get_props
import joblib

add_props = get_props(ds_smiles)
ds_props = np.concatenate([ds_props, add_props], axis=1)

train_smiles, test_smiles, train_props, test_props = train_test_split(ds_smiles, ds_props, test_size=0.2, random_state=42)
train_smiles, valid_smiles, train_props, valid_smiles = train_test_split(train_smiles, train_props, test_size=0.2, random_state=42)

std_sclr = joblib.load('../Transformer/StandardScaler.pkl')
scaled_test_props = std_sclr.transform(test_props)

### **Generation of samples**

In [6]:
def get_samples(model, props, token_to_id, seq_len, start_token='[START]', temperature=1.0, beta=False):
	batch_shape = props.shape[0]
	noise_props = tf.convert_to_tensor(props)

	start_id = token_to_id[start_token]
	dec_tokens = tf.fill([batch_shape, 1], start_id)
	for _ in range(seq_len):
		if type(beta) == float:
			probs = model((noise_props, dec_tokens), beta=beta)
			next_logits = probs[:, -1, :]
			next_logits = tf.argmax(next_logits, axis=-1)
			next_tokens = tf.expand_dims(next_logits, axis=1)
		if temperature == None:
			logits = model((noise_props, dec_tokens))
			next_logits = logits[:, -1, :]
			next_logits = tf.argmax(next_logits, axis=-1)
			next_tokens = tf.expand_dims(next_logits, axis=1)
		if type(temperature) == float:
			logits = model((noise_props, dec_tokens))
			next_logits = logits[:, -1, :]
			next_logits = next_logits / temperature
			next_tokens = tf.random.categorical(next_logits, num_samples=1)
        
		next_tokens = tf.cast(next_tokens, tf.int32)
		dec_tokens = tf.concat([dec_tokens, next_tokens], axis=1)
	
	return dec_tokens
      
beta = 1
test_preds_gan = get_samples(generator, scaled_test_props, token_to_id, SEQ_LEN, beta=beta, temperature=None).numpy()
test_preds_transf = get_samples(transformer_corr_trained, scaled_test_props, token_to_id, SEQ_LEN, beta=False, temperature=None).numpy()
test_preds_transf_rand = get_samples(transformer_corr_trained, scaled_test_props, token_to_id, SEQ_LEN, beta=False, temperature=1.2).numpy()

/home/ubuntu/miniconda3/envs/new_tensorflow_2/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'positional_encoding' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/ubuntu/miniconda3/envs/new_tensorflow_2/lib/python3.12/site-packages/keras/src/ops/nn.py:946: UserWarning: You are using a softmax over axis 3 of a tensor of shape (26167, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/home/ubuntu/miniconda3/envs/new_tensorflow_2/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'positional_encoding_1' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support ma

### **Preprocessing Samples**

In [7]:
import pandas as pd
import selfies as sf

def selfies_to_smiles_and_props(selfies_list):
    smiles_list, props_list = [], []
    for single_selfies in selfies_list:
        single_smiles = sf.decoder(single_selfies)
        props = get_props(pd.Series([single_smiles]))
        smiles_list.append(single_smiles)
        props_list.append(props)
    return smiles_list, np.array(props_list)

special_tokens = ['[PAD]', '[START]', '[END]']
def selfies_preprocessing(selfies_set):
	all_smiles = []
	for selfies in selfies_set:
		single_selfies = ""
		for token in selfies:
			curr_token = id_to_token[str(token)]
			if curr_token not in special_tokens:
				single_selfies += curr_token
		all_smiles.append(single_selfies)
	return all_smiles

test_selfies_gan = selfies_preprocessing(test_preds_gan)
test_selfies_transf = selfies_preprocessing(test_preds_transf)
test_selfies_transf_rand = selfies_preprocessing(test_preds_transf_rand)

test_smiles_gan, test_props_gan = selfies_to_smiles_and_props(test_selfies_gan)
test_smiles_transf, test_props_transf = selfies_to_smiles_and_props(test_selfies_transf)
test_smiles_transf_rand, test_props_transf_rand = selfies_to_smiles_and_props(test_selfies_transf_rand)

## **MSE**

In [8]:
from sklearn.metrics import mean_squared_error

mse_gan = mean_squared_error(y_true=test_props[:, -4:], y_pred=np.squeeze(test_props_gan))
mse_transformer = mean_squared_error(y_true=test_props[:, -4:], y_pred=np.squeeze(test_props_transf))
mse_transformer_randomness = mean_squared_error(y_true=test_props[:, -4:], y_pred=np.squeeze(test_props_transf_rand))

print(f"Gan's Mean Squared Error: {mse_gan:.3f}")
print(f"Transformer's Mean Squared Error: {mse_transformer:.3f}")
print(f"Randomness Transformer's Mean Squared Error: {mse_transformer_randomness:.3f}")

Gan's Mean Squared Error: 33.410
Transformer's Mean Squared Error: 16.913
Randomness Transformer's Mean Squared Error: 34.255
